# Layer-metrics comparison

Loads all `layer_metrics_*.csv` files produced by `run_experiments.py`, filters to a chosen experiment config, and plots algorithm-vs-algorithm comparisons.

Key metric: **`rel_error`** — the fraction of Wanda mass `∑ (W·‖x‖)²` thrown away by pruning. Lower is better. This is the right metric for comparing pruning algorithms because (a) it's the quantity Wanda scores were designed to minimize, and (b) it's normalized so layers of different sizes are comparable.

For each metric of interest we produce **four delta views** (vs a chosen baseline):

| | per layer type | per transformer block |
|---|---|---|
| **aggregated** | bar chart, 1 bar per (layer type × alg), averaged across blocks | line plot, x=block idx, 1 line per alg, averaged across layer types |
| **not aggregated** | strip dots, 7 columns (one per layer type), 1 dot per block | 1 subplot per algorithm, x=block idx, 1 line per layer type |

Time is only shown per transformer block, not aggregated.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

OUTPUT_FOLDER = "benchmark_csvs"

LAYER_TYPES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LAYER_GROUP = {lt: ("Attention" if lt in {"q_proj", "k_proj", "v_proj", "o_proj"} else "MLP")
               for lt in LAYER_TYPES}

# Consistent color per algorithm across all plots
ALG_COLORS = {
    "GA-TETRIS":       "#d62728",
    "Original TETRIS": "#1f77b4",
    "Sort-by-Norm":    "#2ca02c",
    "Block-Wanda":     "#ff7f0e",
    "Random":          "#9467bd",
    "Random-Swaps":    "#8c564b",
}
def alg_color(alg):
    return ALG_COLORS.get(alg, "#555555")

# Consistent color per layer type
LAYER_COLORS = {
    "q_proj":    "#1f77b4",
    "k_proj":    "#ff7f0e",
    "v_proj":    "#2ca02c",
    "o_proj":    "#d62728",
    "gate_proj": "#9467bd",
    "up_proj":   "#8c564b",
    "down_proj": "#e377c2",
}
def layer_color(lt):
    return LAYER_COLORS.get(lt, "#555555")


## 1. Load all CSVs

In [ ]:
csv_files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, "layer_metrics_*.csv")))
print(f"Found {len(csv_files)} CSVs:")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

# Derived columns
df["retained_frac"]     = df["retained_wanda_mass"] / df["total_wanda_mass"]
df["pruned_wanda_mass"] = df["total_wanda_mass"] - df["retained_wanda_mass"]
df["rel_error_weights"] = df["pruned_weight_l1"] / df["total_weight_l1"]
df["block_size"]        = df["block_rows"].astype(str) + "x" + df["block_cols"].astype(str)
df["layer_group"]       = df["layer_type"].map(LAYER_GROUP)

# Map algorithm codenames to thesis display names
ALG_DISPLAY = {
    "our_tetris":                 "GA-TETRIS",
    "original_tetris":            "Original TETRIS",
    "block_wanda":                "Block-Wanda",
    "sort_columns_by_norm":       "Sort-by-Norm",
    "random_permutation_pruning": "Random",
    "random_swaps_find_mask":     "Random-Swaps",
    "random_swaps":               "Random-Swaps",
    "random_swaps_sort_start":    "Random-Swaps (sorted)",
}
df["algorithm"] = df["algorithm"].map(ALG_DISPLAY).fillna(df["algorithm"])

print(f"\nTotal rows: {len(df)}")
print(f"Algorithms:  {sorted(df.algorithm.unique())}")
print(f"Block sizes: {sorted(df.block_size.unique())}")
print(f"Sparsities:  {sorted(df.sparsity.unique())}")


## 2. Pick an experiment config to compare

Each CSV is one algorithm at one `(block_size, sparsity, …)` setting. Fix everything except `algorithm` to get a clean comparison.

In [ ]:
# Filter knobs — edit to pick which experiment slice to compare
FILTER_BLOCK_ROWS = 1
FILTER_BLOCK_COLS = 2
FILTER_SPARSITY   = 0.5

mask = (
    (df.block_rows == FILTER_BLOCK_ROWS) &
    (df.block_cols == FILTER_BLOCK_COLS) &
    (df.sparsity   == FILTER_SPARSITY)
)
df_exp = df[mask].copy()
print(f"Filtered rows: {len(df_exp)}")
print(f"Algorithms in filter: {sorted(df_exp.algorithm.unique())}")

# Baseline to compute deltas against
BASELINE_ALG = "Original TETRIS"
assert BASELINE_ALG in df_exp.algorithm.unique(), \
    f"Baseline '{BASELINE_ALG}' not found. Available: {sorted(df_exp.algorithm.unique())}"


## 3. Helper functions

Four delta views per metric, plus the strip-dot summary used for the algorithm-level comparison.

In [ ]:
def compute_delta(data, metric, baseline_alg):
    """Per-layer (layer_type, layer_idx) delta of `metric` vs baseline algorithm.

    Returns long-format DataFrame with columns: layer_type, layer_idx, algorithm, delta.
    Baseline algorithm is dropped (its delta would be 0 by definition).
    """
    piv = data.pivot_table(
        index=["layer_type", "layer_idx"],
        columns="algorithm",
        values=metric,
        aggfunc="mean",
    )
    delta = piv.subtract(piv[baseline_alg], axis=0).drop(columns=[baseline_alg])
    return (delta.reset_index()
                 .melt(id_vars=["layer_type", "layer_idx"],
                       var_name="algorithm", value_name="delta")
                 .dropna(subset=["delta"]))


# ----------------------------------------------------------------------------
# View 1/4: per layer type, aggregated  (bar chart)
# ----------------------------------------------------------------------------
def delta_layertype_aggregated(data, metric, baseline_alg, ylabel, title_suffix=""):
    """Bar chart: x = layer type, group of bars = algorithms, y = mean delta over all blocks."""
    long = compute_delta(data, metric, baseline_alg)
    agg = (long.groupby(["layer_type", "algorithm"])["delta"]
              .mean().unstack("algorithm")
              .reindex([lt for lt in LAYER_TYPES if lt in long.layer_type.unique()]))

    algs = list(agg.columns)
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(agg.index))
    width = 0.8 / max(len(algs), 1)

    for k, alg in enumerate(algs):
        ax.bar(x + k * width, agg[alg].values, width,
               label=alg, color=alg_color(alg), alpha=0.9)

    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x + width * (len(algs) - 1) / 2)
    ax.set_xticklabels(agg.index)
    ax.set_ylabel(f"Mean Δ {ylabel} vs {baseline_alg}\n(negative = better)")
    ax.set_title(f"Mean Δ {metric} per layer type, averaged across blocks{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# View 2/4: per layer type, not aggregated  (strip dots)
# ----------------------------------------------------------------------------
def delta_layertype_strip(data, metric, baseline_alg, ylabel, title_suffix=""):
    """7 columns (one per layer type); each dot = one transformer block, color = algorithm.

    A horizontal tick marks the per-algorithm mean within each layer type column.
    """
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())
    layer_types_present = [lt for lt in LAYER_TYPES if lt in long.layer_type.unique()]

    fig, ax = plt.subplots(figsize=(14, 6))
    rng_j = np.random.default_rng(0)

    # x slot per layer type; within each slot, sub-positions per algorithm
    n_alg = max(len(algs), 1)
    sub_step = 0.8 / n_alg
    sub_offsets = -0.4 + sub_step / 2 + np.arange(n_alg) * sub_step

    for col_i, lt in enumerate(layer_types_present):
        for a_i, a in enumerate(algs):
            sub = long[(long.layer_type == lt) & (long.algorithm == a)]
            if len(sub) == 0:
                continue
            x_center = col_i + sub_offsets[a_i]
            jitter = rng_j.uniform(-sub_step * 0.35, sub_step * 0.35, size=len(sub))
            ax.scatter(x_center + jitter, sub["delta"].values,
                       color=alg_color(a), alpha=0.65, s=24,
                       edgecolor="white", linewidth=0.4,
                       label=a if col_i == 0 else None)
            # mean tick
            ax.hlines(sub["delta"].mean(),
                      x_center - sub_step * 0.4, x_center + sub_step * 0.4,
                      color=alg_color(a), linewidth=2.2, zorder=3)

    ax.axhline(0, color="black", linewidth=1.1)
    ax.set_xticks(np.arange(len(layer_types_present)))
    ax.set_xticklabels(layer_types_present)
    ax.set_ylabel(f"Δ {ylabel} vs {baseline_alg}\n(negative = better)")
    ax.set_title(f"Per-block Δ {metric} grouped by layer type{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# View 3/4: per transformer block, aggregated  (line plot)
# ----------------------------------------------------------------------------
def delta_block_aggregated(data, metric, baseline_alg, ylabel, title_suffix=""):
    """Line plot: x = block index, y = mean delta across layer types, line per algorithm."""
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())

    agg = (long.groupby(["layer_idx", "algorithm"])["delta"]
              .mean().unstack("algorithm"))

    fig, ax = plt.subplots(figsize=(12, 5))
    for a in algs:
        if a not in agg.columns: continue
        ax.plot(agg.index, agg[a], label=a, color=alg_color(a),
                linewidth=1.6, marker="o", markersize=4, alpha=0.9)
    ax.axhline(0, color='black', linewidth=1.2, label=f"{baseline_alg} (baseline)")
    ax.set_xlabel("Transformer block index")
    ax.set_ylabel(f"Mean Δ {ylabel} across layer types\n(negative = better)")
    ax.set_title(f"Δ {metric} by transformer block, averaged across layer types{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# View 4/4: per transformer block, not aggregated  (one subplot per algorithm)
# ----------------------------------------------------------------------------
def delta_block_per_algorithm(data, metric, baseline_alg, ylabel, title_suffix=""):
    """One subplot per algorithm; x = block idx; one line per layer type.

    Always reserves an extra grid slot for the legend so it lays out cleanly
    regardless of how many algorithms are in the filter.
    """
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())
    if len(algs) == 0:
        print("No non-baseline algorithms in data.")
        return None

    # Reserve one slot for the legend
    n_total = len(algs) + 1
    n_cols = min(3, n_total)
    n_rows = (n_total + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.flatten()

    for k, a in enumerate(algs):
        ax = axes_flat[k]
        sub = long[long.algorithm == a]
        for lt in LAYER_TYPES:
            line = sub[sub.layer_type == lt].sort_values("layer_idx")
            if len(line) == 0:
                continue
            ax.plot(line["layer_idx"], line["delta"],
                    label=lt, color=layer_color(lt),
                    linewidth=1.4, marker="o", markersize=3.5, alpha=0.9)
        ax.axhline(0, color="black", linewidth=1.0)
        ax.set_title(a, fontsize=12, fontweight="bold", color=alg_color(a))
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.set_xlabel("Transformer block index")
        ax.set_ylabel(f"Δ {ylabel}")

    # Hide unused subplots
    for k in range(len(algs), len(axes_flat)):
        axes_flat[k].axis("off")

    # Legend in the first unused slot
    handles = [Line2D([0], [0], color=layer_color(lt), lw=1.8,
                      marker="o", markersize=4, label=lt)
               for lt in LAYER_TYPES if lt in long.layer_type.unique()]
    axes_flat[len(algs)].legend(handles=handles, loc="center", fontsize=11,
                                frameon=False, title="Layer type", title_fontsize=12)

    fig.suptitle(f"Δ {metric} by block, per algorithm{title_suffix}",
                 fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# Strip plot: every layer as a dot, one column per algorithm  (for rel_error and pruned_wanda_mass)
# ----------------------------------------------------------------------------
def strip_delta(data, metric, baseline_alg, ylabel, title_suffix=""):
    """Every layer as a jittered dot; horizontal bar = mean per algorithm."""
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())

    fig, ax = plt.subplots(figsize=(10, 5))
    rng_j = np.random.default_rng(0)
    for k, a in enumerate(algs):
        vals = long[long.algorithm == a]["delta"].values
        if len(vals) == 0:
            continue
        jitter = rng_j.uniform(-0.25, 0.25, size=len(vals))
        ax.scatter(k + jitter, vals, color=alg_color(a), alpha=0.55,
                   s=22, edgecolor="white", linewidth=0.4)
        ax.hlines(vals.mean(), k - 0.35, k + 0.35,
                  color=alg_color(a), linewidth=2.5, zorder=3)

    ax.axhline(0, color='black', linewidth=1.1)
    ax.set_xticks(range(len(algs)))
    ax.set_xticklabels(algs, rotation=15, ha='right')
    ax.set_ylabel(f"Δ {ylabel} vs {baseline_alg}\n(negative = better)")
    ax.set_title(f"Per-layer Δ {metric}, each dot = one layer{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    fig.tight_layout()
    return fig


# ----------------------------------------------------------------------------
# Time view: per transformer block, not aggregated, no delta (absolute time)
# ----------------------------------------------------------------------------
def time_block_per_algorithm(data, title_suffix=""):
    """Absolute time per block; one subplot per algorithm; line per layer type.

    Always reserves an extra grid slot for the legend.
    """
    algs = sorted(data.algorithm.unique())
    n_total = len(algs) + 1
    n_cols = min(3, n_total)
    n_rows = (n_total + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.flatten()

    for k, a in enumerate(algs):
        ax = axes_flat[k]
        sub = data[data.algorithm == a]
        for lt in LAYER_TYPES:
            line = sub[sub.layer_type == lt].sort_values("layer_idx")
            if len(line) == 0:
                continue
            ax.plot(line["layer_idx"], line["total_time_sec"],
                    label=lt, color=layer_color(lt),
                    linewidth=1.4, marker="o", markersize=3.5, alpha=0.9)
        ax.set_title(a, fontsize=12, fontweight="bold", color=alg_color(a))
        ax.set_yscale("log")
        ax.grid(True, linestyle="--", alpha=0.35, which="both")
        ax.set_xlabel("Transformer block index")
        ax.set_ylabel("Pruning time (s, log scale)")

    for k in range(len(algs), len(axes_flat)):
        axes_flat[k].axis("off")

    handles = [Line2D([0], [0], color=layer_color(lt), lw=1.8,
                      marker="o", markersize=4, label=lt)
               for lt in LAYER_TYPES if lt in data.layer_type.unique()]
    axes_flat[len(algs)].legend(handles=handles, loc="center", fontsize=11,
                                frameon=False, title="Layer type", title_fontsize=12)

    fig.suptitle(f"Pruning time per block, per algorithm{title_suffix}",
                 fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    return fig


## 4. Summary tables

One row per algorithm, then per-layer-type breakdown of mean rel_error.

In [ ]:
# Overall per-algorithm summary
overall = (df_exp.groupby("algorithm")
           .agg(mean_rel_error=("rel_error", "mean"),
                median_rel_error=("rel_error", "median"),
                mean_retained_frac=("retained_frac", "mean"),
                mean_time_sec=("total_time_sec", "mean"),
                total_time_sec=("total_time_sec", "sum"),
                n_layers=("rel_error", "count"))
           .sort_values("mean_rel_error"))
print("=== Overall (all layers pooled) ===")
display(overall.round(5))


In [ ]:
# Per-algorithm × per-layer-type: mean rel_error
by_type = (df_exp.groupby(["layer_type", "algorithm"])["rel_error"]
           .mean().unstack("algorithm"))
by_type = by_type.reindex([lt for lt in LAYER_TYPES if lt in by_type.index])
print("=== Mean rel_error per layer type × algorithm (lower is better) ===")
display(by_type.round(5))

# Highlight best algorithm per layer type
def _highlight_min(row):
    is_min = row == row.min()
    return ["font-weight: bold; background-color: #d4f4dd" if v else "" for v in is_min]
display(by_type.round(5).style.apply(_highlight_min, axis=1))


## 5. Relative error (Wanda-weighted): four delta views

`rel_error = ‖ΔW · ‖x‖‖² / ‖W · ‖x‖‖²` — the fraction of Wanda mass thrown away.

In [ ]:
SUFFIX = f"  —  block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY}  (vs {BASELINE_ALG})"

# View 1/4: per layer type, aggregated
delta_layertype_aggregated(df_exp, "rel_error", BASELINE_ALG,
                           "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 2/4: per layer type, not aggregated
delta_layertype_strip(df_exp, "rel_error", BASELINE_ALG,
                      "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 3/4: per transformer block, aggregated
delta_block_aggregated(df_exp, "rel_error", BASELINE_ALG,
                       "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 4/4: per transformer block, not aggregated
delta_block_per_algorithm(df_exp, "rel_error", BASELINE_ALG,
                          "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


## 6. Pruned Wanda mass (absolute): four delta views

`pruned_wanda_mass = total_wanda_mass − retained_wanda_mass` — absolute Wanda score thrown away. Same shape as rel_error but unnormalized — useful for spotting large layers carrying disproportionate weight.

In [ ]:
# View 1/4: per layer type, aggregated
delta_layertype_aggregated(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                           "pruned Wanda mass", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 2/4: per layer type, not aggregated
delta_layertype_strip(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                      "pruned Wanda mass", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 3/4: per transformer block, aggregated
delta_block_aggregated(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                       "pruned Wanda mass", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 4/4: per transformer block, not aggregated
delta_block_per_algorithm(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                          "pruned Wanda mass", title_suffix=SUFFIX); plt.show()


## 7. Strip-dot summary

Each dot is one layer; horizontal bar is the per-algorithm mean. Quickest "did my algorithm beat the baseline" view.

In [ ]:
strip_delta(df_exp, "rel_error", BASELINE_ALG,
            "rel_error (fraction)", title_suffix=SUFFIX); plt.show()
strip_delta(df_exp, "pruned_wanda_mass", BASELINE_ALG,
            "pruned Wanda mass", title_suffix=SUFFIX); plt.show()


## 8. Weight-space relative error: four delta views

`rel_error_weights = ‖ΔW‖₁ / ‖W‖₁` — fraction of L1 weight mass thrown away, ignoring activations. Handy sanity check that Wanda-weighted improvements aren't coming from just removing tiny weights.

In [ ]:
# View 1/4: per layer type, aggregated
delta_layertype_aggregated(df_exp, "rel_error_weights", BASELINE_ALG,
                           "weight-space rel_error", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 2/4: per layer type, not aggregated
delta_layertype_strip(df_exp, "rel_error_weights", BASELINE_ALG,
                      "weight-space rel_error", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 3/4: per transformer block, aggregated
delta_block_aggregated(df_exp, "rel_error_weights", BASELINE_ALG,
                       "weight-space rel_error", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 4/4: per transformer block, not aggregated
delta_block_per_algorithm(df_exp, "rel_error_weights", BASELINE_ALG,
                          "weight-space rel_error", title_suffix=SUFFIX); plt.show()


## 9. Pruned weight L1 (absolute): four delta views

`pruned_weight_l1 = ‖ΔW‖₁` — absolute L1 of removed weights, no normalization.

In [ ]:
# View 1/4: per layer type, aggregated
delta_layertype_aggregated(df_exp, "pruned_weight_l1", BASELINE_ALG,
                           "pruned weight L1", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 2/4: per layer type, not aggregated
delta_layertype_strip(df_exp, "pruned_weight_l1", BASELINE_ALG,
                      "pruned weight L1", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 3/4: per transformer block, aggregated
delta_block_aggregated(df_exp, "pruned_weight_l1", BASELINE_ALG,
                       "pruned weight L1", title_suffix=SUFFIX); plt.show()


In [ ]:
# View 4/4: per transformer block, not aggregated
delta_block_per_algorithm(df_exp, "pruned_weight_l1", BASELINE_ALG,
                          "pruned weight L1", title_suffix=SUFFIX); plt.show()


## 10. Pruning time

Per transformer block, not aggregated, **absolute** (not delta — comparing across orders of magnitude). One subplot per algorithm, line per layer type, log y-scale.

In [ ]:
time_block_per_algorithm(df_exp, title_suffix=SUFFIX); plt.show()


## 11. Compare across block sizes (single algorithm)

If multiple block sizes are present in the data, sweep block size for one chosen algorithm.

In [ ]:
# sort_columns_by_norm, block_wanda, original_tetris, our_tetris, random_swaps
FOCUS_ALG = ALG_DISPLAY["our_tetris"]

block_sizes = sorted(df[df.algorithm == FOCUS_ALG].block_size.unique(),
                     key=lambda s: (int(s.split("x")[0]), int(s.split("x")[1])))

if len(block_sizes) > 1:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
    axes_flat = axes.flatten()
    cmap = plt.cm.viridis(np.linspace(0, 0.85, len(block_sizes)))

    for i, lt in enumerate(LAYER_TYPES):
        ax = axes_flat[i]
        for bs, color in zip(block_sizes, cmap):
            sub = df[(df.algorithm == FOCUS_ALG) & (df.layer_type == lt) & (df.block_size == bs)]
            sub = sub.sort_values("layer_idx")
            ax.plot(sub["layer_idx"], sub["rel_error"],
                    label=f"block {bs}", color=color, linewidth=1.4, marker="o", markersize=3.5)
        ax.set_title(lt, fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.35)
        if i % 4 == 0: ax.set_ylabel("Rel. error")
        if i >= 4:     ax.set_xlabel("Transformer block index")

    axes_flat[7].axis("off")
    handles = [Line2D([0], [0], color=c, lw=1.8, marker="o", markersize=4, label=f"block {bs}")
               for bs, c in zip(block_sizes, cmap)]
    axes_flat[7].legend(handles=handles, loc="center", fontsize=11, frameon=False,
                        title=f"{FOCUS_ALG}", title_fontsize=12)
    fig.suptitle(f"Block-size sweep for {FOCUS_ALG}", fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    plt.show()
else:
    print(f"Only one block size ({block_sizes[0]}) in data for {FOCUS_ALG} — nothing to sweep.")
